# 04 · LPJmL source aggregation

Area-weight annual ISIMIP2b LPJmL `pirrww` on the frozen SPAM 0.5° grid. Original design: 48 historical + 96 future `.nc4` files, four climate forcings, 12 direct crops. The output is a per-forcing diagnostic, not the final 56-product release.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'

import re
import xarray as xr
WEIGHTS=RUNS/'spam-weights'
HIST=DATA/'isimip'/'raw'/'historical'
FUT=DATA/'isimip'/'raw'/'future'
OUT=RUNS/'lpjml'
FORCINGS={'GFDL-ESM2M':'gfdl-esm2m','HadGEM2-ES':'hadgem2-es',
          'IPSL-CM5A-LR':'ipsl-cm5a-lr','MIROC5':'miroc5'}
HIST_YEARS=[2000,2005]
FUT_YEARS=[2010,2015,2020]

In [ ]:
def file_metadata(path,period):
    stem=path.name.lower()
    if not stem.startswith('lpjml_'):raise ValueError(path)
    crop=re.search(r'pirrww-([a-z0-9]+)-firr',stem)
    if crop is None:raise ValueError(f'Crop not encoded in {path.name}')
    forcing=[k for k,v in FORCINGS.items() if f'_{v}_' in stem]
    if len(forcing)!=1:raise ValueError(f'Ambiguous climate forcing {path.name}')
    pathway='HISTORICAL' if period=='historical' else (
        'RCP26' if '_rcp26_' in stem else 'RCP60' if '_rcp60_' in stem else None)
    if pathway is None:raise ValueError(f'Unrecognised pathway {path.name}')
    return {'crop_code':crop.group(1),'climate_forcing':forcing[0],
            'climate_scenario':pathway,'impact_model':'LPJmL','period_type':period,
            'source_file':path.name}

def years_from_source(ds):
    time=ds['time']
    units=str(time.attrs.get('units',''))
    m=re.search(r'years since\s+(\d{4})',units.lower())
    if not m:raise ValueError(f'Unexpected ISIMIP time units: {units}')
    values=np.asarray(time.values,dtype=float)
    if not np.allclose(values,np.round(values),atol=1e-8):
        raise ValueError('Source uses a non-integer annual time axis')
    return m and (int(m.group(1))+values.astype(int))

def country_weighted(da_year,weights,metadata):
    lat=xr.DataArray(weights.lpjml_lat.to_numpy(float),dims='points')
    lon=xr.DataArray(weights.lpjml_lon.to_numpy(float),dims='points')
    values=np.asarray(da_year.sel(lat=lat,lon=lon).values,dtype=float)
    if values.ndim!=1 or len(values)!=len(weights):raise ValueError('NetCDF grid alignment failed')
    w=weights.copy()
    w['pirrww_mm_yr']=values
    w['valid']=np.isfinite(values)
    w['represented_irrigated_ha_cell']=np.where(w.valid,w.corrected_irrigated_ha,0)
    w['weighted_mm_ha']=np.where(w.valid,values*w.corrected_irrigated_ha,0)
    keys=['country_iso3','year','fable_product_norm','lpjml_crop_code']
    result=w.groupby(keys,as_index=False).agg(total_irrigated_ha=('corrected_irrigated_ha','sum'),
        represented_irrigated_ha=('represented_irrigated_ha_cell','sum'),
        weighted_mm_ha_sum=('weighted_mm_ha','sum'),grid_cells=('valid','size'),
        valid_grid_cells=('valid','sum'))
    result['spatial_coverage_fraction']=np.where(result.total_irrigated_ha>0,
        result.represented_irrigated_ha/result.total_irrigated_ha,np.nan)
    result['weighted_pirrww_mm_yr']=np.where(result.represented_irrigated_ha>0,
        result.weighted_mm_ha_sum/result.represented_irrigated_ha,np.nan)
    result['represented_water_km3']=result.weighted_mm_ha_sum*10/1e9
    result['qc_status']=np.select([result.represented_irrigated_ha<=0,
       result.spatial_coverage_fraction<0.99],['NO_COVERAGE','REVIEW_COVERAGE'],default='PASS')
    for key,val in metadata.items():result[key]=val
    return result.drop(columns='weighted_mm_ha_sum')

In [ ]:
source_files={
 'historical':sorted(HIST.rglob('lpjml_*.nc4')),
 'future':sorted(FUT.rglob('lpjml_*.nc4'))}
assert len(source_files['historical'])==48, 'Original inventory requires 48 historical LPJmL files'
assert len(source_files['future'])==96, 'Original inventory requires 96 future LPJmL files'
weights={
 'historical':pd.read_parquet(WEIGHTS/'00_FABLE_SPAM_corrected_irrigated_area_05deg_2000_2005.parquet'),
 'future':pd.read_parquet(WEIGHTS/'00_FABLE_SPAM_corrected_irrigated_area_05deg_2010_2015_2020.parquet')}
outputs=[]
for period,files in source_files.items():
    for file in files:
        meta=file_metadata(file,period)
        subset=weights[period][weights[period].lpjml_crop_code==meta['crop_code']]
        if subset.empty:raise ValueError(f'Source lacks SPAM direct-crop support: {file.name}')
        with xr.open_dataset(file,decode_times=False,mask_and_scale=True) as ds:
            candidates=[k for k in ds.data_vars if 'pirrww' in k.lower()]
            if len(candidates)!=1:raise ValueError(f'Unambiguous PIRRWW variable needed: {file.name}')
            years=years_from_source(ds)
            target=HIST_YEARS if period=='historical' else FUT_YEARS
            for year in target:
                idx=np.flatnonzero(years==year)
                if len(idx)!=1:raise ValueError(f'{file.name} missing/duplicate {year}')
                one=subset[subset.year==year]
                outputs.append(country_weighted(ds[candidates[0]].isel(time=int(idx[0])),one,meta))
raw=pd.concat(outputs,ignore_index=True)
print(raw.groupby(['impact_model','climate_scenario','year']).size())

In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
file=OUT/'06_GLOBAL_LPJmL_pirrww_SPAM_weighted_RAW_2000_2020.csv.gz'
if file.exists():raise FileExistsError(file)
raw.to_csv(file,index=False)
print('LPJmL forcing-level output:',file)